Questo notebook analizza il dataset test_dataset2 costruito nel Notebook01 per capire se è adatto alla costruzione di un sistema di raccomandazione.
In particolare analizza sparsità, distribuzione per categoria e possibili soglie di filtraggio.

## 1-setup

In [ ]:
#Se mancano pacchetti, esegui questa cella una sola volta.
%pip install -q "datasets==3.6.0" pandas pyarrow requests tqdm
%pip install matplotlib

In [ ]:
from pathlib import Path
import json
import math
import warnings #gestire warnings
import numpy as np 
import pandas as pd
import pyarrow.parquet as pq 
import matplotlib.pyplot as plt 

warnings.filterwarnings("ignore", category=FutureWarning) #nasconde avvisi per avere output meno rumoroso
#impostazioni di pandas
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")  

## 2 percorsi file

In [ ]:
#percorsi
CATEGORY="AmazonLike_v1"
DATASET_NAME="test_dataset2"

def find_project_root(start:Path=Path.cwd())->Path:
    for candidate in [start,*start.parents]:
        if (candidate/"data").exists() or (candidate/"notebooks").exists() or (candidate/".git").exists():
            return candidate.resolve()
    return start.resolve()

PROJECT_ROOT=find_project_root() #determina automaticamente la cartella principale del progetto
PROCESSED_DIR=PROJECT_ROOT/"data"/"processed"/f"{DATASET_NAME}_intersection" #percorso dati notebook01
ANALYSIS_DIR=PROCESSED_DIR/"analysis" #dove salvare risultati notebook02
ANALYSIS_DIR.mkdir(parents=True,exist_ok=True) #la crea se non esiste già

#percorsi dei file prodotti nel notebook01
ITEMS_PATH=PROCESSED_DIR/"items.parquet"
QA_PATH=PROCESSED_DIR/"qa.parquet"
RATINGS_PATH=PROCESSED_DIR/"ratings.parquet"
SUMMARY_PATH=PROCESSED_DIR/"summary.json"
CATEGORY_SUMMARY_PATH=PROCESSED_DIR/"category_summary.csv"

#dizionario per associare nome leggibile ad ogni percorso
paths={
    "items":ITEMS_PATH,
    "qa":QA_PATH,
    "ratings":RATINGS_PATH,
    "summary":SUMMARY_PATH,
    "category_summary":CATEGORY_SUMMARY_PATH,
}
#controllo dei percorsi
print("Project root:",PROJECT_ROOT)
print("Processed dir:",PROCESSED_DIR)
print("Analysis dir:",ANALYSIS_DIR)
for name,path in paths.items():
    print(f"{name:16s}->{path} | exists={path.exists()}")

## 3 lettura dati parquet

In [ ]:
#lettura dei metadata dei parquet
def parquet_info(path: Path)->dict:
    pf=pq.ParquetFile(path)
    return{
        "file":path.name,
        "rows":pf.metadata.num_rows,
        "row_groups":pf.metadata.num_row_groups,
        "columns": len(pf.schema_arrow.names),
        "column_names": ", ".join(pf.schema_arrow.names),
    }

parquet_infos=pd.DataFrame([
    parquet_info(ITEMS_PATH),
    parquet_info(QA_PATH),
    parquet_info(RATINGS_PATH),
])

parquet_infos

In [ ]:
summary=json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
display(pd.DataFrame([summary]).T.rename(columns={0: "rows_from_summary_json"}))

## 4 caricamento dei dati

In [ ]:
#caricamento dei dati
def parquet_columns(path: Path)-> list[str]: #funzione che restituisce i nomi delle colonne
    return pq.ParquetFile(path).schema_arrow.names

def columns_keep(path: Path, wanted: list[str])-> list[str]: #funzione che tiene solo le colonne richieste
    available=set(parquet_columns(path))
    return [col for col in wanted if col in available]

items_df=pd.read_parquet(ITEMS_PATH)
qa_df=pd.read_parquet(QA_PATH)

wanted_rating_columns=[
    "user_id","parent_asin","source_category","hf_category","category_group","asin","rating","timestamp","verified_purchase",
    "review_title","review_text",#se presenti le colonne testuali, in base a cosa si è scelto nel notebook 01
]

rating_columns=columns_keep(RATINGS_PATH,wanted_rating_columns)
ratings_df=pd.read_parquet(RATINGS_PATH,columns=rating_columns)

print("items_df:", items_df.shape)
print("qa_df:", qa_df.shape)
print("ratings_df:", ratings_df.shape)
print("Colonne rating caricate:", rating_columns)

In [ ]:
display(items_df.head(3))
display(qa_df.head(3))
display(ratings_df.head(3))

## 5 controlli di coerenza

In [ ]:
#controlli generali di coerenza
def pct(numerator, denominator): #funzione che useremo anche dopo per calcolare la percentuale
    if denominator == 0:
        return np.nan
    return numerator/denominator *100

items_set=set(items_df["parent_asin"].dropna()) #insieme parentasin nei metadati, droppando valori nulli
ratings_items_set=set(ratings_df["parent_asin"].dropna()) #insieme parentasin dei rating
qa_items_set=set(qa_df["parent_asin"].dropna()) #insieme parentasin qa
#dizionario con statistiche generali del dataset
basic_stats={
    "num_product_items": items_df["parent_asin"].nunique(), #conta quanti prodotti unici nella tabella items
    "num_rating_rows": len(ratings_df), #conta quante righe di rating presenti
    "num_users":ratings_df["user_id"].nunique(dropna=True), #quanti utenti unici compaiono nei rating
    "num_products_with_ratings": ratings_df["parent_asin"].nunique(dropna=True), #quanti prodotti distinti hanno almeno un rating
    "num_qa_rows":len(qa_df), #quante righe di domande/risposte presenti
    "num_products_with_qa": qa_df["parent_asin"].nunique(dropna=True), #quanti prodotti distinti hanno almeno una domanda/risposta
    "items_parent_asin_unique": bool(items_df["parent_asin"].is_unique), #controlla se parent_asin è unico nella tabella items
    "ratings_products_all_in_items": ratings_items_set.issubset(items_set), #verifica se tutti i prodotti nei rating sono presenti anche nei metadati items
    "qa_products_all_in_items": qa_items_set.issubset(items_set), #controlla se tutti prodotti qa sono presenti anche nei metadati items
    "items_with_rating_pct": pct(len(items_set & ratings_items_set), len(items_set)), #calcola percentuale di prodotti items che hanno almeno un rating
    "items_with_qa_pct": pct(len(items_set & qa_items_set), len(items_set)), #calcola percentuale di prodotti items che hanno almeno una riga QA
}

basic_stats_df=pd.DataFrame.from_dict(basic_stats, orient="index", columns=["value"]) #converte dizionario delle stats generali in tabella pandas
display(basic_stats_df)

basic_stats_df.to_csv(ANALYSIS_DIR/"notebook02_basic_stats.csv")

In [ ]:
#analisi per categoria del dataset multi-categoria
#serve a controllare se una categoria domina troppo o se alcune categorie restano piccole
if "source_category" in items_df.columns:
    category_items_df=items_df.groupby("source_category").agg(
        products=("parent_asin","nunique"),
        avg_metadata_rating=("avg_rating","mean"),
        avg_metadata_num_ratings=("num_ratings","mean"),
    ).reset_index()
else:
    category_items_df=pd.DataFrame()

if "source_category" in ratings_df.columns:
    category_ratings_df=ratings_df.groupby("source_category").agg(
        rating_rows=("rating","size"),
        users=("user_id","nunique"),
        products_with_ratings=("parent_asin","nunique"),
        avg_rating=("rating","mean"),
    ).reset_index()
else:
    category_ratings_df=pd.DataFrame()

if "source_category" in qa_df.columns:
    category_qa_df=qa_df.groupby("source_category").agg(
        qa_rows=("parent_asin","size"),
        products_with_qa=("parent_asin","nunique"),
    ).reset_index()
else:
    category_qa_df=pd.DataFrame()

category_profile_df=category_items_df
if not category_ratings_df.empty:
    category_profile_df=category_profile_df.merge(category_ratings_df,on="source_category",how="outer")
if not category_qa_df.empty:
    category_profile_df=category_profile_df.merge(category_qa_df,on="source_category",how="outer")

if CATEGORY_SUMMARY_PATH.exists():
    category_summary_from_nb01_df=pd.read_csv(CATEGORY_SUMMARY_PATH)
    print("Riepilogo prodotto dal Notebook01")
    display(category_summary_from_nb01_df)

print("Profilo categorie dal dataset caricato")
display(category_profile_df.sort_values("products",ascending=False) if "products" in category_profile_df.columns else category_profile_df)
category_profile_df.to_csv(ANALYSIS_DIR/"notebook02_category_profile.csv",index=False)

In [ ]:
#se alcuni prodotti sono presenti nei ratings ma non in items (se esce 0 perfetto)
missing_rating_products = ratings_items_set - items_set
#se alcuni prodotti sono presenti nel QA ma assenti in items (se esce 0 perfetto)
missing_qa_products = qa_items_set - items_set
print("Prodotti in ratings ma non in items:", len(missing_rating_products))
print("Prodotti in qa ma non in items:", len(missing_qa_products))
if missing_rating_products:
    print("Esempi ratings fuori items:", list(missing_rating_products)[:5])
if missing_qa_products:
    print("Esempi QA fuori items:", list(missing_qa_products)[:5])

## 6 controllo qualità dei dati

In [ ]:
#ora bisogna controllare la qualità di questi dati
important_columns={ #dizionario con colonne importanti da controllare
    "items":["parent_asin","source_category","title","category"],
    "qa":["parent_asin","source_category","question","answer"],
    "ratings":["user_id","parent_asin","source_category","rating"],
}
null_rows=[]
#scorriamo le tabelle e colonne da controllare
for df_name, cols in important_columns.items():
    df={"items":items_df, "qa":qa_df, "ratings":ratings_df}[df_name]
    for col in cols:
        if col in  df.columns:
            null_rows.append({
                "table":df_name,
                "column":col,
                "null_count":int(df[col].isna().sum()), #conta quanti valori nulli presenti nella colonna
                "null_pct": pct(int(df[col].isna().sum()), len(df)), #conta la percentuale di valori nulli nella colonna
            })

nulls_df=pd.DataFrame(null_rows)
nulls_df


In [ ]:
#vediamo se ci sono rating nulli
invalid_rating_mask= ratings_df["rating"].isna() | ~ratings_df["rating"].between(1,5)
invalid_ratings=ratings_df.loc[invalid_rating_mask]
print("Rating nulli o fuori scala:", len(invalid_ratings))
if len(invalid_ratings)>0:
    display(invalid_ratings.head()) 


In [ ]:
#calcoliamo distribuzione dei voti
rating_dist=(
    ratings_df["rating"].value_counts(dropna=False).sort_index().rename_axis("rating").reset_index(name="count")
)
rating_dist["pct"]=rating_dist["count"]/len(ratings_df)*100 #percentuale di ogni voto sul tot
rating_dist

In [ ]:
#schema dei dati visti sopra sui rating
plt.figure(figsize=(7,4))
rating_dist.dropna(subset=["rating"]).plot(
    x="rating",y="count",kind="bar",legend=False,ax=plt.gca()
)
plt.title("Distribuzione dei rating nel dataset")
plt.xlabel("Rating")
plt.ylabel("Num di valutazioni")
plt.tight_layout()
plt.show()

In [ ]:
#funzione per capire se timestamp sono in sec o millisec
def timestamp_un(series:pd.Series)->str | None:
    values=pd.to_numeric(series.dropna(), errors="coerce").dropna() #converte timestamp in numeri
    if values.empty:
        return None
    median_value=values.median()
    if median_value>1e12: #se compatibile con millisecondi
        return "ms"
    if median_value>1e9: #se compatibile con secondi
        return "s"
    return None
#controlla se esiste colonna timestamp
if "timestamp" in ratings_df.columns:
    timestamp_unit=timestamp_un(ratings_df["timestamp"]) #stima l'unità del timestamp
    print("Unità stimata del timestamp: ",timestamp_unit)

    if timestamp_unit is not None:
        ratings_df["datetime"]=pd.to_datetime(
            ratings_df["timestamp"], unit=timestamp_unit,errors="coerce" #passa a pandas la colonna timestamp da convertire
        )
        print("data minima: ",ratings_df["datetime"].min())
        print("data massima: ",ratings_df["datetime"].max())
        display(ratings_df[["timestamp","datetime"]].head())
    else:
        print("timestamp esiste ma unità non riconosciuta")
else:
    print("Colonna timestamp assente, non puoi usarlo come criterio")

In [ ]:
#controlliamo duplicati userid+parentasin
pair_columns=["user_id","parent_asin"]
duplicate_mask=ratings_df.duplicated(pair_columns,keep=False)#crea maschera che marca tutte le righe appartenenti a coppie utente-prodotto duplicate
num_duplicate_rows=int(duplicate_mask.sum()) #conta righe duplicate
num_duplicate_pairs=int(ratings_df.loc[duplicate_mask,pair_columns].drop_duplicates().shape[0]) #conta coppie duplicate
print("Righe coinvolte in duplicati user_id+parent_asin:", num_duplicate_rows)
print("Coppie duplicate distinte:", num_duplicate_pairs)

if num_duplicate_rows>0:
    display( #seleziona solo righe duplicate, e le ordina
        ratings_df.loc[duplicate_mask].sort_values(pair_columns+(["timestamp"] if "timestamp" in ratings_df.columns else [])).head(10)
    )


In [ ]:
#andiamo ad analizzare le valutazioni per ogni anno
if "datetime" in ratings_df.columns:
    ratings_each_year =(
        ratings_df["datetime"]
        .dt.year
        .value_counts()
        .sort_index()
        .rename_axis("year")
        .reset_index(name="count")
    )

    ratings_each_year["pct"]=ratings_each_year["count"]/len(ratings_df)*100
    display(ratings_each_year)
    plt.figure(figsize=(10,4))
    plt.bar(ratings_each_year["year"], ratings_each_year["count"])
    plt.title("Distribuzione dei rating per ogni anno")
    plt.xlabel("Anno")
    plt.ylabel("Numero di rating")
    plt.tight_layout()
    plt.show()

## 7 costruzione tabella nuova pulita

Costruiamo la tabella sistemata:
rimuoviamo le righe non valide, duplicate.

In [ ]:
def build_int_table(ratings: pd.DataFrame) ->pd.DataFrame: #funzione per pulire e togliere duplicati
    base=ratings.copy() #crea copia
    base=base.dropna(subset=["user_id","parent_asin","rating"]) #rimuove righe senza utente, prodotto, rating
    base=base[base["rating"].between(1,5)].copy() #mantiene rating fra 1 e 5

    if "timestamp" in base.columns:
        base=base.sort_values(["user_id","parent_asin","timestamp"]) #ordina valutazioni per utente, prodotto e timestamp
        interactions=base.drop_duplicates(["user_id","parent_asin"],keep="last").copy()#per ogni coppia tiene solo l'ultima
    else:
        interactions=( #come fallback nel caso di assenza di timestamp usa media dei rating per coppia utente-prodotto
            base.groupby(["user_id","parent_asin"], as_index=False).agg(rating=("rating","mean"))
        )
    return interactions.reset_index(drop=True) #restituisce interazioni pulite con indice ricreato da zero

interactions_df=build_int_table(ratings_df) #applica la funzione ai rating

print("Righe rating originali: ",f"{len(ratings_df):,}")
print("Interazioni dopo la pulizia:",f"{len(interactions_df):,}")
print("Righe rimosse nella pulizia:",f"{len(ratings_df)-len(interactions_df):,}")
display(interactions_df.head())

## 8 analisi degli utenti

Analizziamo gli utenti, quante interazioni ha ciascun utente

In [ ]:
def analyze_quantilies(counts: pd.Series, name: str)-> pd.DataFrame: #funzione per riassumere conteggi
    quantiles=counts.quantile([0.25,0.50,0.75,0.90,0.95,0.99]).to_dict()
    data={
        "entity":name,
        "num_entities":counts.shape[0],
        "mean":counts.mean(),
        "median:":counts.median(),
        "min":counts.min(),
        "p25":quantiles.get(0.25),
        "p75":quantiles.get(0.75),
        "p90":quantiles.get(0.90),
        "p95":quantiles.get(0.95),
        "p99":quantiles.get(0.99),
        "max":counts.max(),
    }
    return pd.DataFrame([data])

user_counts=interactions_df["user_id"].value_counts()
user_summary=analyze_quantilies(user_counts, "users")
user_summary

In [ ]:
user_thresholds=[1,2,3,4,5,6,7,8,9,10,20,50] #soglie da testare minime di interazione per ogni utente
rows=[]

for t in user_thresholds:
    valid_users=user_counts[user_counts>=t].index #.index prende solo identificativi degli utenti rimasti
    filtered=interactions_df[interactions_df["user_id"].isin(valid_users)] #filtriamo tabella, tenendo solo utenti validi che rispettano la soglia di interazioni

    rows.append({
        "min_interactions":t,
        "num_users":filtered["user_id"].nunique(), #num utenti unici rimasti
        "num_products":filtered["parent_asin"].nunique(), #num prodotti unici rimasti
        "num_interactions":len(filtered), #num interazioni rimaste
        "percentage_users":filtered["user_id"].nunique()/interactions_df["user_id"].nunique()*100,
        "percentage_interactions":len(filtered)/len(interactions_df)*100
    })

user_threshold_table=pd.DataFrame(rows)
user_threshold_table

In [ ]:
plt.figure(figsize=(7,4))
counts=user_counts[user_counts>0]
#crea 50 intervalli su scala logaritmica tra 10^0=1 e max num di interazioni=10^2=100 circa (in realtà 98)
#essendo molto sbilanciata, tanti utenti hanno poche interazioni, pochi utenti ne hanno molte.super
spaces=np.logspace(0,np.log10(counts.max()),50) if counts.max()>1 else 1 
plt.hist(counts,bins=spaces)
plt.xscale("log")
plt.yscale("log")
plt.title("Distribuzione interazione per utente")
plt.xlabel("Num di interazioni di quell'utente")
plt.ylabel("Num utenti")
plt.tight_layout()
plt.show()

## 9 analisi dei prodotti
vogliamo evitare prodotti con pochissimi rating, ma anche evitare filtri troppo aggressivi

In [ ]:
item_counts=interactions_df["parent_asin"].value_counts() #contiamo quante interazioni ha ogni prodotto
item_summary=analyze_quantilies(item_counts, "products") #riepilogo della distribuzione
item_summary

In [ ]:
#soglie di thresholds da testare
item_thresholds=[1,2,3,4,5,6,7,8,9,10,20,50,100]
rows=[]
for t in item_thresholds:
    valid_items=item_counts[item_counts>=t].index
    filtered=interactions_df[interactions_df["parent_asin"].isin(valid_items)]
    rows.append({ #aggiungiamo dizionario con le statistiche sulla soglia corrente
        "min_product_interactions":t,
        "num_users":filtered["user_id"].nunique(),
        "num_products_remaining":filtered["parent_asin"].nunique(),
        "num_interactions":len(filtered),
        "percentage_users":filtered["user_id"].nunique()/interactions_df["user_id"].nunique()*100,
        "percentage_products":filtered["parent_asin"].nunique()/interactions_df["parent_asin"].nunique()*100,
        "percentage_Interactions":len(filtered)/len(interactions_df)*100
    })

item_thresholds_table=pd.DataFrame(rows)
item_thresholds_table

In [ ]:
plt.figure(figsize=(7,4))
counts=item_counts[item_counts>0]
spaces=np.logspace(0,np.log10(counts.max()),50) if counts.max()>1 else 1 
plt.hist(counts,bins=spaces)
plt.xscale("log")
plt.yscale("log")
plt.title("Distribuzione interazione per prodotto")
plt.xlabel("Num di interazioni di quel prodotto")
plt.ylabel("Num prodotti")
plt.tight_layout()
plt.show()

## 10 analisi QA per prodotto
inizialmente potrebbe non essere aggiunto, ma è utile vedere ora quante domande/risposte sono associate ai prodotti

In [ ]:
qa_counts=qa_df["parent_asin"].value_counts() #conta quante righe associate a ciascun prodotto
total_products=items_df["parent_asin"].nunique() #calcola n tot di prodotti nel catalogo

products_with_qa=qa_counts.shape[0] #num prodotti con almeno una riga qa
total_qa_rows=len(qa_df)

qa_overview=pd.DataFrame([{
    "tot_products":total_products,#num tot di prodotti
    "products_with_qa":products_with_qa,
    "percentage_with_qa":products_with_qa/total_products*100,
    "total_qa_rows":total_qa_rows,
    "mean-qa_perproducts":qa_counts.mean(),
    "median-qa_perproducts":qa_counts.median(),
    "max-qa_perproduct":qa_counts.max()
}])
qa_summary=analyze_quantilies(qa_counts, "products_with_qa")
display(qa_overview)
display(qa_summary)

In [ ]:
#definiamo soglie minime delle righe di QA
qa_thresholds=[1,2,3,4,5,6,7,8,9,10,20,50]
rows=[]

for t in qa_thresholds:
    valid_qa_items=qa_counts[qa_counts>=t].index
    qa_filtered=qa_df[qa_df["parent_asin"].isin(valid_qa_items)]#filtriamo tabella tenendo solo righe dei prodotti validi per questa thresholds
    interactions_filtered=interactions_df[
        interactions_df["parent_asin"].isin(valid_qa_items)#filtriamo le interazioni rating degli stessi prodotti per capire se sono validi

    ]
    rows.append({
        "minqa_rows":t,
        "num_products":len(valid_qa_items), #quanti prodotti rimangono
        "numqa_rows":len(qa_filtered), #quante righe qa rimangono
        "numusers_withratings":interactions_filtered["user_id"].nunique(), #quanti utenti hanno rating su quei prodotti
        "num_interactions":len(interactions_filtered),#quante interazioni rating rimangono
        "percentage_products":len(valid_qa_items)/items_df["parent_asin"].nunique() *100,  # %prodotti mantenuti
        "percentage_qarows":len(qa_filtered)/len(qa_df)*100, # %righe qa mantenute
        "percentage_interactions":len(interactions_filtered)/len(interactions_df)*100 # % interazioni rating mantenuti
    })

qa_thresholds_table=pd.DataFrame(rows)
qa_thresholds_table

## 11 Profilo prodotti rating + qa + metadati
creiamo una vista sintetica per analisi

In [ ]:
profile_col=[
    col for col in  ["parent_asin","source_category","category_group","title","category","avg_rating","num_ratings","price","store"] #scorre colonne candidate
    if col in items_df.columns #list comprehension, crea lista con colonne che esistono davvero dentro dentro l'altra lista
]

item_profile=items_df[profile_col].copy()#crea tab con solo quelle colonne
item_profile=item_profile.merge(
    item_counts.rename("interaction_count"), #rinomina conteggi dei rating
    left_on="parent_asin",  #merge usa parentasin come chiave nel dataframe di sx
    right_index=True, #merge usa indice della serie di destra come chiave
    how="left", #left join per mantenere tutti i prodotti anche se non hanno quel conteggio
)

item_profile=item_profile.merge(
    qa_counts.rename("qa_count"),
    left_on="parent_asin",
    right_index=True,
    how="left",
)
item_profile["interaction_count"] =item_profile["interaction_count"].fillna(0).astype(int) #sostituisce mancanti con 0 e converte a interi
item_profile["qa_count"]= item_profile["qa_count"].fillna(0).astype(int) #sostituisce mancanti con 0 e converte a interi
item_profile.sort_values("interaction_count",ascending=False).head(20) #mostra ordinati dal piu interagito al meno

## 12 sparsità della matrice utente-prodotto

In [ ]:
#funzione per calcoalre sparsità della matrice
def sparsity_statistics(df: pd.DataFrame, user_col="user_id", item_col="parent_asin")->dict:
    num_users=df[user_col].nunique() #conta n utenti unici
    num_items=df[item_col].nunique()#num prodotti unici
    num_interactions=len(df) #num interazioni
    possible_pairs=num_users*num_items #coppie possibili nella matrice completa
    density=num_interactions/possible_pairs if possible_pairs else np.nan #densità come interazioni osservate/coppie possibili

    return{
        "num_users":num_users,
        "num_items":num_items,
        "num_interactions":num_interactions,
        "possible_pairs":possible_pairs,
        "density":density,
        "density_percentage": density*100 if not pd.isna(density) else np.nan, 
        "sparsity_percentage": (1-density)*100 if not pd.isna(density) else np.nan, #sparsità come 1-densità
    }
base_sparsity=sparsity_statistics(interactions_df)
base_sparsity_df=pd.DataFrame([base_sparsity])
base_sparsity_df

## 13 confronto di configurazioni candidate

In [ ]:
#funz per applicare filtro utenti-prodotti
def func_filter(
    df: pd.DataFrame,
    min_user_int:int, #soglia minima interazione per utente
    min_item_int:int, #soglia minima per interazioni prodotto
    user_col:str="user_id",
    item_col:str="parent_asin",
    max_iter:int=50, #max iterazioni, evitare loop
)->tuple[pd.DataFrame,pd.DataFrame]:
    current=df.copy()
    history=[]
    for step in range(1,max_iter+1):
        #salviamo num precedenti
        rows_before=len(current)
        users_before=current[user_col].nunique()
        items_before=current[item_col].nunique()
        if min_item_int>1 and len(current)>0: #se la soglia prodotto è maggiore di 1 e la tabella non è vuota, filtra i prodotti
            item_counts_step=current[item_col].value_counts() #conta interazioni
            valid_items=item_counts_step[item_counts_step>=min_item_int].index #seleziona prodotti
            current=current[current[item_col].isin(valid_items)] #tiene solo interazioni dei prodotti che abbiamo tenuto
        if min_user_int>1 and len(current)>0: #se la soglia utente >1 e tab non vuota
            user_counts_step=current[user_col].value_counts()
            valid_users=user_counts_step[user_counts_step>=min_user_int].index 
            current=current[current[user_col].isin(valid_users)]
        #contiamo num dopo filtraggio
        rows_after=len(current)
        users_after=current[user_col].nunique()
        items_after=current[item_col].nunique()
        history.append({ #aggiunge alla storia le statistiche dell'iterazione
            "iteration":step,
            "min_user_int":min_user_int,
            "min_item_int":min_item_int,
            "rows_before":rows_before,
            "rows_after":rows_after,
            "rows_removed":rows_before-rows_after,
            "users_before":users_before,
            "users_after":users_after,
            "users_removed":users_before-users_after,
            "items_before":items_before,
            "items_after":items_after,
            "items_removed":items_before-items_after,
            
        })
        if rows_after==rows_before: #stoppa se non ci sono righe da rimuovere
            break
    return current.reset_index(drop=True), pd.DataFrame(history) #ritorna dataframe filtrato e la storia delle iterazioni

In [ ]:
user_thresholds_grid=[2,3,4,5]
item_thresholds_grid=[5,6,7,8,9,10,20]
candidate_rows=[] #lista dove salveremo una riga di risultati per ogni combinazione
candidate_histories={} #diz dove salveremo la history di ogni config

#num tot prima
total_users=interactions_df["user_id"].nunique()
total_items=interactions_df["parent_asin"].nunique()
total_interactions=len(interactions_df)

for min_u in user_thresholds_grid:
    for min_i in item_thresholds_grid:
        conf_used=f"user{min_u}_item{min_i}" #nome della configurazione usata ora
        filtered_df,history_df=func_filter( #applichiamo filtro
            df=interactions_df,
            min_user_int=min_u,
            min_item_int=min_i,
            user_col="user_id",
            item_col="parent_asin",
            max_iter=50
        )
        candidate_histories[conf_used]=history_df #salviamo history
        #num tot dopo
        num_users=filtered_df["user_id"].nunique()
        num_items=filtered_df["parent_asin"].nunique()
        num_interactions=len(filtered_df)
        possible_pairs=num_users*num_items
        #calcolo densità e numeri medi interazioni per prodotti e per utenti
        density=num_interactions/possible_pairs if possible_pairs>0 else 0
        avg_interactions_peruser=num_interactions/num_users if num_users>0 else 0
        avg_interactions_peritem=num_interactions/num_items if num_items>0 else 0
        candidate_rows.append({ #salviamo le statistiche della configurazione corrente
            "config_name":conf_used,
            "min_user_int":min_u,
            "min_item_int":min_i,
            "num_users":num_users,
            "num_items":num_items,
            "num_interactions":num_interactions,
            "pct_users":num_users/total_users*100,
            "pct_items":num_items/total_items*100,
            "pct_interactions":num_interactions/total_interactions*100,
            "density_pct":density*100,
            "avg_interactions_peruser":avg_interactions_peruser,
            "avg_interactions_peritem":avg_interactions_peritem,
            "iterations":len(history_df),
            "supports_train_val_test":min_u>=3 #1 per training, 1 per val, 1 per test, teoricamente lo teniamo come minimo
        })

candidate_filters_df=pd.DataFrame(candidate_rows)
candidate_filters_df=candidate_filters_df.sort_values(
    ["min_user_int","min_item_int"]
).reset_index(drop=True)
candidate_filters_df



In [ ]:
candidate_filters_path=ANALYSIS_DIR/"notebook02_candidate_filter_comparison.csv"
candidate_filters_df.to_csv(candidate_filters_path,index=False)
print("Tabella salvata in:",candidate_filters_path)

## 14 scelta della configurazione
Scegliamo utenti con interazioni >=3 e prodotti con interazioni >=8

In [ ]:
SELECTED_CONFIG="user3_item8" #puoi cambiare per sceglierne altre

if SELECTED_CONFIG not in set(candidate_filters_df["config_name"]):
    raise ValueError(f"Configurazione non trovata: {SELECTED_CONFIG}")


selected_config_row=candidate_filters_df[
    candidate_filters_df["config_name"]==SELECTED_CONFIG
].iloc[0] #recuperiamo riga della config scelta

selected_min_user_int=int(selected_config_row["min_user_int"])
selected_min_item_int=int(selected_config_row["min_item_int"])

selected_interactions_df,selected_history_df=func_filter( #applichiamo filtro per ottenere dataset candidato per il prossimo notebook03
    df=interactions_df,
    min_user_int=selected_min_user_int,
    min_item_int=selected_min_item_int,
    user_col="user_id",
    item_col="parent_asin",
    max_iter=50,
)

print("Configurazione scelta:",SELECTED_CONFIG)
print("Utenti rimasti nel dataset:",selected_interactions_df["user_id"].nunique())
print("Prodotti rimasti nel dataset:",selected_interactions_df["parent_asin"].nunique())
print("Interazioni rimaste:",len(selected_interactions_df))
display(selected_history_df)
display(pd.DataFrame([selected_config_row]))

In [ ]:
selected_sparsity=sparsity_statistics(selected_interactions_df) #sparsità
selected_sparsity_df=pd.DataFrame([selected_sparsity]) #la convertiamo per mostrarla in tabella

sparsity_comp_df=pd.concat(
    [
        base_sparsity_df.assign(dataset="completo"),
        selected_sparsity_df.assign(dataset=SELECTED_CONFIG)
    ],
    ignore_index=True,
)
sparsity_comp_df=sparsity_comp_df[
    ["dataset","num_users","num_items","num_interactions","possible_pairs","density_percentage","sparsity_percentage"]
]
sparsity_comp_df

In [ ]:
selected_summary={
    "selected_config_name":SELECTED_CONFIG,
    "min_user_interactions":selected_min_user_int,
    "min_item_interactions":selected_min_item_int,
    "num_users":int(selected_interactions_df["user_id"].nunique()),
    "num_items":int(selected_interactions_df["parent_asin"].nunique()),
    "num_interactions":int(len(selected_interactions_df)),
    "density_pct":float(selected_sparsity["density_percentage"]),
    "sparsity_pct":float(selected_sparsity["sparsity_percentage"]),
    "category":CATEGORY,
    "dataset_name":DATASET_NAME,
}
selected_config_path=ANALYSIS_DIR/"notebook02_selected_filter_config.json" #salviamo in file json la configurazione scelta
selected_config_path.write_text(
    json.dumps(selected_summary,indent=2), encoding="utf-8", #convertiamo dizionario in testo json formattato
)
print("Configurazione scelta salvata in:",selected_config_path)
selected_summary

## 15 dataset candidati

In [ ]:
selected_products=selected_interactions_df["parent_asin"].unique() #elenco prodotti rimasti nelle interazioni filtrate
selected_items_df=items_df[
    items_df["parent_asin"].isin(selected_products)
].copy() #filtriamo la tabella prodotti tenendo solo quelli rimasti nel dataset candidato

selected_qa_df=qa_df[
    qa_df["parent_asin"].isin(selected_products)
].copy() #filtriamo qa tenendo solo prodotti rimasti nel dataset candidato

# i 3 path in cui salvare
selected_interactions_path=PROCESSED_DIR/f"candidate_interactions_{SELECTED_CONFIG}.parquet"
selected_items_path=PROCESSED_DIR/f"candidate_items_{SELECTED_CONFIG}.parquet"
selected_qa_path=PROCESSED_DIR/f"candidate_qa_{SELECTED_CONFIG}.parquet"
#salviamo
selected_interactions_df.to_parquet(selected_interactions_path,index=False)
selected_items_df.to_parquet(selected_items_path,index=False)
selected_qa_df.to_parquet(selected_qa_path,index=False)

print("Interazioni del dataset candidato salvate in:",selected_interactions_path)
print("Prodotti del dataset candidato salvati in:",selected_items_path)
print("QA dei prodotti candidati salvati in:",selected_qa_path)

print("selected_interactions_df:",selected_interactions_df.shape)
print("selected_items_df:",selected_items_df.shape)
print("selected_qa_df:",selected_qa_df.shape)